# SC2 Replay Data Verification & Visualization Notebook

**Role in the larger system:** This notebook provides interactive verification and
visualization tools for the parquet files produced by the SC2 replay extraction
pipeline (`src_new/`). It helps developers manually inspect lifecycle events,
correlations, and data sparsity to ensure the extraction pipeline is producing
correct, complete output.

**Sections:**
1. Setup & Data Loading
2. Lifecycle Event Table (most important)
3. Interactive Entity Explorer
4. Unit/Building Timeline Chart
5. Correlation Heatmap
6. Data Sparsity Visualization

---
## Section 1: Setup & Data Loading

Imports all required libraries, auto-discovers parquet files, loads the game
state data, and parses column names into an entity catalog using the pipeline's
regex convention (`ENTITY_COL_RE`).

In [ ]:
"""
Section 1: Setup & Data Loading

Imports core libraries, attempts to import ipywidgets for interactive controls,
auto-discovers parquet files, loads the first game_state file, prints shape
info, and builds an entity catalog by parsing all column names.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import re
from pathlib import Path
from collections import defaultdict

# Attempt to import ipywidgets for interactive exploration (Section 3).
# If not installed, we gracefully degrade to callable functions.
WIDGETS_AVAILABLE = False
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    WIDGETS_AVAILABLE = True
    print("ipywidgets loaded successfully -- interactive explorer will be available.")
except ImportError:
    from IPython.display import display
    print("ipywidgets not installed -- Section 3 will use callable functions instead.")
    print("Install with: pip install ipywidgets")

# ---- Regex from shared_constants.py for parsing entity column names ----
# Groups: (1) player prefix "p1"/"p2", (2) middle = botname_entitytype,
#         (3) 3-digit sequence ID, (4) attribute name
ENTITY_COL_RE = re.compile(r"^(p[12])_(.+)_(\d{3})_(.+)$")

# ---- Lifecycle strings from shared_constants.py ----
# These are the string values that appear IN entity attribute columns
# when a lifecycle transition occurs (instead of numeric data).
#
# NOTE: "existing" and "under_construction" are valid lifecycle states in
# the extractors, but they write REAL numeric data to the attribute columns
# (not string markers), so they never appear as strings in the parquet.
# "started" (without prefix) is never produced by any extractor -- the
# extractors emit "unit_started" and "building_started" instead.
# "building" is a transitional state that also writes real data.
# These phantom entries have been removed to avoid confusion.
# (See diagnostics/023-lifecycle-timeline-rca.md)
ALL_LIFECYCLE_STRINGS = frozenset({
    "unit_started", "building_started", "completed",
    "destroyed", "cancelled",
})

In [ ]:
# ---- Auto-discover parquet files ----
parquet_dir = Path("../../data/quickstart/parquet")
parquet_files = sorted(parquet_dir.glob("*_game_state.parquet"))

if not parquet_files:
    raise FileNotFoundError(
        f"No *_game_state.parquet files found in {parquet_dir.resolve()}"
    )

print(f"Found {len(parquet_files)} game state parquet file(s):")
for pf in parquet_files:
    print(f"  {pf.name}")

# Load the first file (change index to load a different one)
PARQUET_PATH = parquet_files[0]
df = pd.read_parquet(PARQUET_PATH)

print(f"\nLoaded: {PARQUET_PATH.name}")
print(f"  Rows (game loops): {len(df):,}")
print(f"  Columns:           {len(df.columns):,}")
if "timestamp_seconds" in df.columns:
    duration_min = df["timestamp_seconds"].max() / 60.0
    print(f"  Duration:          {duration_min:.1f} minutes")

In [ ]:
def build_entity_catalog(columns):
    """
    Parse all column names and build a catalog of entities.

    Each entity is identified by its full prefix (player_middle_seqid),
    and we record the entity type (extracted from middle), player,
    sequence ID, and the set of attributes present for that entity.

    Parameters:
        columns (list[str]): List of DataFrame column names.

    Returns:
        dict: Mapping entity_prefix -> {
            'player': str,
            'middle': str (botname_entitytype),
            'entity_type': str (last segment of middle),
            'seq_id': str (3-digit),
            'attributes': set[str],
            'columns': list[str] (full column names)
        }

    Depends on / calls:
        - ENTITY_COL_RE (module-level compiled regex)
    """
    catalog = {}

    for col in columns:
        m = ENTITY_COL_RE.match(col)
        if not m:
            continue

        player, middle, seq_id, attribute = m.groups()
        # Entity type is the last underscore-separated segment of middle.
        # SC2 type names never contain underscores after sanitization.
        entity_type = middle.rsplit("_", 1)[-1]
        entity_prefix = f"{player}_{middle}_{seq_id}"

        if entity_prefix not in catalog:
            catalog[entity_prefix] = {
                "player": player,
                "middle": middle,
                "entity_type": entity_type,
                "seq_id": seq_id,
                "attributes": set(),
                "columns": [],
            }

        catalog[entity_prefix]["attributes"].add(attribute)
        catalog[entity_prefix]["columns"].append(col)

    return catalog


# Build the catalog from the loaded DataFrame
entity_catalog = build_entity_catalog(df.columns)

print(f"Total entities in catalog: {len(entity_catalog)}")

# Count entities by player and type
type_counts = defaultdict(lambda: defaultdict(int))
for prefix, info in entity_catalog.items():
    type_counts[info["player"]][info["entity_type"]] += 1

for player in sorted(type_counts):
    total = sum(type_counts[player].values())
    print(f"\n{player}: {total} entities")
    for etype, count in sorted(type_counts[player].items(), key=lambda x: -x[1]):
        print(f"  {etype}: {count}")

---
## Section 2: Lifecycle Event Table (MOST IMPORTANT)

Scans all entity columns for lifecycle string values and produces a clean
DataFrame of every lifecycle transition in the game. This is the primary
verification tool: it lets you see exactly when each entity was started,
completed, destroyed, or cancelled.

In [ ]:
def extract_lifecycle_events(df, entity_catalog):
    """
    Scan all entity columns for lifecycle string values and build a DataFrame
    of every lifecycle transition event in the game.

    Key insight: All attribute columns for a given entity have the SAME lifecycle
    string at the SAME game_loop, so we only need to check ONE column per entity
    (the first column in each entity's column list).

    Parameters:
        df (pd.DataFrame): The game state DataFrame.
        entity_catalog (dict): Output from build_entity_catalog().

    Returns:
        pd.DataFrame: Columns = ['entity_name', 'entity_type', 'player',
                       'event_type', 'game_loop', 'timestamp_seconds']
                       Sorted by game_loop ascending.

    Depends on / calls:
        - entity_catalog from build_entity_catalog()
        - ALL_LIFECYCLE_STRINGS (module-level frozenset)
    """
    events = []

    for entity_prefix, info in entity_catalog.items():
        # Only need to check ONE column per entity -- all attribute columns
        # share the same lifecycle string at any given game_loop.
        representative_col = info["columns"][0]
        col_data = df[representative_col]

        # Find rows where the value is a lifecycle string.
        # Entity columns are object/string dtype with mixed string/numeric data.
        for idx in col_data.dropna().index:
            val = col_data.iloc[idx]
            # Check if the value is one of the known lifecycle strings
            if isinstance(val, str) and val in ALL_LIFECYCLE_STRINGS:
                game_loop = df["game_loop"].iloc[idx]
                timestamp = df["timestamp_seconds"].iloc[idx]
                events.append({
                    "entity_name": entity_prefix,
                    "entity_type": info["entity_type"],
                    "player": info["player"],
                    "event_type": val,
                    "game_loop": game_loop,
                    "timestamp_seconds": timestamp,
                })

    # Build DataFrame and sort by game_loop
    if not events:
        return pd.DataFrame(
            columns=["entity_name", "entity_type", "player",
                     "event_type", "game_loop", "timestamp_seconds"]
        )

    lifecycle_df = pd.DataFrame(events)
    lifecycle_df = lifecycle_df.sort_values("game_loop").reset_index(drop=True)
    return lifecycle_df


# Extract all lifecycle events
lifecycle_events = extract_lifecycle_events(df, entity_catalog)

print(f"Total lifecycle events: {len(lifecycle_events)}")
print(f"\nEvent type counts:")
print(lifecycle_events["event_type"].value_counts().to_string())
print(f"\nFirst 20 events:")
display(lifecycle_events.head(20))

In [ ]:
def show_entity_lifecycle(lifecycle_df, entity_pattern):
    """
    Filter lifecycle events by a text pattern and display all lifecycle
    transitions for matching entities in a readable table.

    Parameters:
        lifecycle_df (pd.DataFrame): Output from extract_lifecycle_events().
        entity_pattern (str): Substring to match against entity_name, entity_type,
                              or player (e.g., "marine", "p1", "barracks").

    Returns:
        pd.DataFrame: Filtered lifecycle events (also displayed inline).

    Depends on / calls:
        - extract_lifecycle_events() output (lifecycle_df parameter)
    """
    pattern_lower = entity_pattern.lower()

    # Match against entity_name, entity_type, or player
    mask = (
        lifecycle_df["entity_name"].str.lower().str.contains(pattern_lower, na=False)
        | lifecycle_df["entity_type"].str.lower().str.contains(pattern_lower, na=False)
        | lifecycle_df["player"].str.lower().str.contains(pattern_lower, na=False)
    )

    filtered = lifecycle_df[mask].copy()

    if filtered.empty:
        print(f"No lifecycle events matching '{entity_pattern}'")
        return filtered

    print(f"Lifecycle events matching '{entity_pattern}': {len(filtered)} events")
    print(f"Unique entities: {filtered['entity_name'].nunique()}")
    display(filtered)
    return filtered


# Example: show lifecycle for all marines
_ = show_entity_lifecycle(lifecycle_events, "marine")

In [ ]:
# Show lifecycle for buildings
_ = show_entity_lifecycle(lifecycle_events, "barracks")

---
## Section 3: Interactive Entity Explorer

If ipywidgets is available, provides a dropdown + text filter to interactively
browse entities and see their lifecycle events, data values at key game_loops,
and raw string values at transitions.

If ipywidgets is NOT available, equivalent callable functions are provided.

In [ ]:
def explore_entity(df, entity_catalog, lifecycle_events, entity_prefix):
    """
    Display detailed information about a single entity, including its lifecycle
    events, data values at key game_loops, and raw string values at transitions.

    Parameters:
        df (pd.DataFrame): The game state DataFrame.
        entity_catalog (dict): Output from build_entity_catalog().
        lifecycle_events (pd.DataFrame): Output from extract_lifecycle_events().
        entity_prefix (str): The entity prefix to explore (e.g., "p1_veterran_another_marine_001").

    Returns:
        None (prints/displays information inline).

    Depends on / calls:
        - build_entity_catalog() output
        - extract_lifecycle_events() output
    """
    if entity_prefix not in entity_catalog:
        print(f"Entity '{entity_prefix}' not found in catalog.")
        # Show similar matches
        matches = [k for k in entity_catalog if entity_prefix.lower() in k.lower()]
        if matches:
            print(f"Similar entities: {matches[:10]}")
        return

    info = entity_catalog[entity_prefix]
    print(f"={'=' * 60}")
    print(f"Entity: {entity_prefix}")
    print(f"  Type:       {info['entity_type']}")
    print(f"  Player:     {info['player']}")
    print(f"  Seq ID:     {info['seq_id']}")
    print(f"  Attributes: {sorted(info['attributes'])}")
    print(f"  # Columns:  {len(info['columns'])}")
    print()

    # --- Lifecycle events for this entity ---
    entity_events = lifecycle_events[
        lifecycle_events["entity_name"] == entity_prefix
    ]
    print(f"Lifecycle Events ({len(entity_events)}):")
    if not entity_events.empty:
        display(entity_events[["event_type", "game_loop", "timestamp_seconds"]])
    else:
        print("  (no lifecycle events found)")
    print()

    # --- Data values at key game_loops (first non-NaN, lifecycle transitions, last non-NaN) ---
    representative_col = info["columns"][0]
    col_data = df[representative_col].dropna()

    if col_data.empty:
        print("No data (all NaN).")
        return

    # Collect key game_loop indices to display
    key_indices = set()
    # First and last non-NaN rows
    key_indices.add(col_data.index[0])
    key_indices.add(col_data.index[-1])
    # Lifecycle transition rows
    for idx in col_data.index:
        val = col_data.loc[idx]
        if isinstance(val, str) and val in ALL_LIFECYCLE_STRINGS:
            key_indices.add(idx)
            # Also include the row immediately after the transition
            next_idx_pos = col_data.index.get_loc(idx)
            if next_idx_pos + 1 < len(col_data.index):
                key_indices.add(col_data.index[next_idx_pos + 1])

    key_indices = sorted(key_indices)

    # Build a small DataFrame of values at key game_loops across all entity columns
    key_data_rows = []
    for idx in key_indices:
        row = {"game_loop": df["game_loop"].iloc[idx]}
        for col in info["columns"]:
            # Extract just the attribute name for a shorter column label
            m = ENTITY_COL_RE.match(col)
            attr_name = m.group(4) if m else col
            row[attr_name] = df[col].iloc[idx]
        key_data_rows.append(row)

    key_df = pd.DataFrame(key_data_rows)
    print(f"Data at key game_loops ({len(key_df)} rows):")
    # Transpose for readability when there are many attributes
    display(key_df.set_index("game_loop").T)


# Sorted list of entity prefixes for dropdown/selection
sorted_entity_names = sorted(entity_catalog.keys())

In [ ]:
if WIDGETS_AVAILABLE:
    # ---- Interactive widget-based entity explorer ----

    # Text filter to narrow entity list
    filter_text = widgets.Text(
        value="",
        placeholder="Type to filter (e.g., marine, p1, barracks)",
        description="Filter:",
        layout=widgets.Layout(width="400px"),
    )

    # Dropdown to select a specific entity
    entity_dropdown = widgets.Dropdown(
        options=sorted_entity_names,
        description="Entity:",
        layout=widgets.Layout(width="500px"),
    )

    # Output area for entity details
    explorer_output = widgets.Output()

    def on_filter_change(change):
        """
        Update the dropdown options when the filter text changes.

        Parameters:
            change: ipywidgets change dict with 'new' key for current text.

        Returns:
            None (updates dropdown widget in-place).

        Depends on / calls:
            - sorted_entity_names (module-level list)
        """
        filter_val = change["new"].lower()
        if filter_val:
            filtered = [n for n in sorted_entity_names if filter_val in n.lower()]
        else:
            filtered = sorted_entity_names
        entity_dropdown.options = filtered

    def on_entity_select(change):
        """
        Display entity details when a new entity is selected from dropdown.

        Parameters:
            change: ipywidgets change dict with 'new' key for selected entity.

        Returns:
            None (displays output in explorer_output widget).

        Depends on / calls:
            - explore_entity() function
        """
        with explorer_output:
            clear_output(wait=True)
            if change["new"]:
                explore_entity(df, entity_catalog, lifecycle_events, change["new"])

    filter_text.observe(on_filter_change, names="value")
    entity_dropdown.observe(on_entity_select, names="value")

    print("Interactive Entity Explorer")
    print("Use the filter box to narrow the entity list, then select an entity.")
    display(widgets.VBox([filter_text, entity_dropdown, explorer_output]))

else:
    # ---- Fallback: callable functions ----
    print("No ipywidgets -- use explore_entity() directly:")
    print("  explore_entity(df, entity_catalog, lifecycle_events, 'p1_..._marine_001')")
    print()
    print("Available entity prefixes (first 20):")
    for name in sorted_entity_names[:20]:
        print(f"  {name}")
    if len(sorted_entity_names) > 20:
        print(f"  ... and {len(sorted_entity_names) - 20} more.")

In [ ]:
# Example: manually explore a specific entity (works with or without widgets)
# Pick the first marine entity if one exists, otherwise the first entity
marine_entities = [n for n in sorted_entity_names if "marine" in n.lower()]
example_entity = marine_entities[0] if marine_entities else sorted_entity_names[0]
explore_entity(df, entity_catalog, lifecycle_events, example_entity)

---
## Section 4: Unit/Building Timeline Chart

Gantt-chart-like visualization showing entity lifespans.
- X-axis: game_loop (or timestamp_seconds)
- Y-axis: each entity (grouped by player, then type)
- Green marker at "started", Blue at "completed", Red at "destroyed"

In [ ]:
# ── Configurable Parameters: Entity Timeline ─────────────
# All visual constants for plot_entity_timeline() are defined here so they
# can be tweaked in one place without modifying the function body.
TIMELINE_FIGURE_WIDTH = 16          # figure width in inches (height auto-calculated)
TIMELINE_FONT_SIZE_TITLE = 16       # title font size
TIMELINE_FONT_SIZE_LABELS = 12      # axis label font size (x-axis label)
TIMELINE_FONT_SIZE_TICKS = 10       # x-axis tick label font size
TIMELINE_FONT_SIZE_YTICKS = 7       # y-axis tick label font size (entity names)
TIMELINE_FONT_SIZE_LEGEND = 8       # legend entry font size
TIMELINE_MARGIN = dict(left=0.08, right=0.95, top=0.92, bottom=0.10)

# Few's categorical palette — muted colors that convey meaning without shouting
TIMELINE_COLOR_STARTED = "#59a14f"   # muted green  (Few: green)
TIMELINE_COLOR_COMPLETED = "#4e79a7" # muted blue   (Few: blue)
TIMELINE_COLOR_DESTROYED = "#e15759" # muted red    (Few: red)
TIMELINE_COLOR_CANCELLED = "#f28e2b" # muted orange (Few: orange)
TIMELINE_COLOR_LIFESPAN = "#cccccc"  # light gray connecting line
TIMELINE_COLOR_OTHER = "#9c755f"     # muted brown for unexpected event types
TIMELINE_COLOR_BACKGROUND = "white"  # figure and axes background

TIMELINE_SPINE_COLOR = "#888888"     # thin gray for left/bottom spines
TIMELINE_GRID_ALPHA = 0.2           # very subtle grid
TIMELINE_GRID_COLOR = "#cccccc"     # light gray gridlines
TIMELINE_MARKER_SIZE = 45           # scatter marker area (points²)
TIMELINE_MARKER_EDGE_WIDTH = 0.3    # thin marker border (less chartjunk)
TIMELINE_MARKER_EDGE_COLOR = "#888888"  # gray edge instead of black
TIMELINE_LINE_WIDTH_LIFESPAN = 1.5  # lifespan connector line width
TIMELINE_DPI = 150                  # figure resolution for display


def plot_entity_timeline(
    lifecycle_df,
    player_filter=None,
    type_filter=None,
    max_entities=40,
    use_timestamp=False,
    figsize=(16, None),
):
    """
    Plot a Gantt-chart-like timeline of entity lifecycles, styled with
    Tufte/Few data visualization principles (minimal chartjunk, white
    background, muted categorical colors, light grid).

    Each entity gets a horizontal row. Lifecycle events are shown as colored
    markers using Few's categorical palette: green for started, blue for
    completed, red for destroyed, orange for cancelled. A light gray line
    connects the first and last event for each entity to show its lifespan.

    Parameters:
        lifecycle_df (pd.DataFrame): Output from extract_lifecycle_events().
        player_filter (str or None): If set, only show entities for this player
                                     (e.g., "p1" or "p2").
        type_filter (str or None): If set, only show entities whose entity_type
                                   contains this substring (e.g., "marine").
        max_entities (int): Maximum number of entities to display (to keep
                            the chart readable). Defaults to 40.
        use_timestamp (bool): If True, use timestamp_seconds for x-axis;
                              otherwise use game_loop. Defaults to False.
        figsize (tuple): (width, height) for the figure. If height is None,
                         it is calculated from the number of entities.

    Returns:
        matplotlib.figure.Figure: The created figure.

    Depends on / calls:
        - extract_lifecycle_events() output (lifecycle_df parameter)
        - TIMELINE_* configurable constants (module-level)
    """
    filtered = lifecycle_df.copy()

    # Apply player filter
    if player_filter:
        filtered = filtered[filtered["player"] == player_filter]

    # Apply type filter (substring match)
    if type_filter:
        type_lower = type_filter.lower()
        filtered = filtered[
            filtered["entity_type"].str.lower().str.contains(type_lower, na=False)
        ]

    if filtered.empty:
        print("No events match the filter criteria.")
        return None

    # Choose x-axis column based on user preference
    x_col = "timestamp_seconds" if use_timestamp else "game_loop"
    x_label = "Timestamp (seconds)" if use_timestamp else "Game Loop"

    # Get unique entities, sorted by player then type then seq_id for grouping
    unique_entities = (
        filtered.drop_duplicates("entity_name")
        .sort_values(["player", "entity_type", "entity_name"])
        ["entity_name"]
        .tolist()
    )

    # Limit to max_entities for readability
    if len(unique_entities) > max_entities:
        print(
            f"Showing first {max_entities} of {len(unique_entities)} entities. "
            f"Use type_filter or player_filter to narrow results."
        )
        unique_entities = unique_entities[:max_entities]
        filtered = filtered[filtered["entity_name"].isin(unique_entities)]

    # Map entity names to y-axis positions (one row per entity)
    entity_to_y = {name: i for i, name in enumerate(unique_entities)}
    num_entities = len(unique_entities)

    # Auto-calculate figure height if not provided, scaling with entity count
    fig_width = figsize[0] if figsize[0] is not None else TIMELINE_FIGURE_WIDTH
    fig_height = figsize[1] if figsize[1] is not None else max(4, num_entities * 0.35)

    # ── Figure Setup (Tufte: white background, no chartjunk) ─────────
    fig, ax = plt.subplots(
        figsize=(fig_width, fig_height),
        dpi=TIMELINE_DPI,
        facecolor=TIMELINE_COLOR_BACKGROUND,
    )
    ax.set_facecolor(TIMELINE_COLOR_BACKGROUND)
    fig.subplots_adjust(**TIMELINE_MARGIN)

    # ── Remove Chartjunk: hide top/right spines, style remaining ones ─
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(TIMELINE_SPINE_COLOR)
    ax.spines["bottom"].set_color(TIMELINE_SPINE_COLOR)
    ax.spines["left"].set_linewidth(0.5)
    ax.spines["bottom"].set_linewidth(0.5)

    # Style tick marks to match the muted spine color
    ax.tick_params(axis="x", colors=TIMELINE_SPINE_COLOR, width=0.5, length=3)
    ax.tick_params(axis="y", colors=TIMELINE_SPINE_COLOR, width=0.5, length=3)

    # ── Color mapping using Few's categorical palette (muted tones) ──
    # Each event type maps to a semantically appropriate muted color.
    # Only event types that actually appear as string markers in the parquet
    # are included. "existing", "under_construction", and "building" write
    # real numeric data (not strings) so they never appear in lifecycle
    # event extraction. "started" (without prefix) is never produced by
    # any extractor. (See diagnostics/023-lifecycle-timeline-rca.md)
    event_colors = {
        "completed": TIMELINE_COLOR_COMPLETED,         # muted blue
        "destroyed": TIMELINE_COLOR_DESTROYED,          # muted red
        "cancelled": TIMELINE_COLOR_CANCELLED,          # muted orange
        "unit_started": TIMELINE_COLOR_STARTED,         # muted green
        "building_started": TIMELINE_COLOR_STARTED,     # muted green (same as unit)
    }

    # ── Draw lifespan lines connecting first and last event per entity ─
    for entity_name in unique_entities:
        entity_events = filtered[filtered["entity_name"] == entity_name]
        if len(entity_events) >= 2:
            x_start = entity_events[x_col].min()
            x_end = entity_events[x_col].max()
            y_pos = entity_to_y[entity_name]
            ax.plot(
                [x_start, x_end],
                [y_pos, y_pos],
                color=TIMELINE_COLOR_LIFESPAN,
                linewidth=TIMELINE_LINE_WIDTH_LIFESPAN,
                zorder=1,
            )

    # ── Plot event markers ───────────────────────────────────────────
    for _, event in filtered.iterrows():
        entity_name = event["entity_name"]
        if entity_name not in entity_to_y:
            continue
        y_pos = entity_to_y[entity_name]
        x_val = event[x_col]
        color = event_colors.get(event["event_type"], TIMELINE_COLOR_OTHER)
        ax.scatter(
            x_val,
            y_pos,
            color=color,
            s=TIMELINE_MARKER_SIZE,
            zorder=2,
            edgecolors=TIMELINE_MARKER_EDGE_COLOR,
            linewidth=TIMELINE_MARKER_EDGE_WIDTH,
        )

    # ── Y-axis: entity names ─────────────────────────────────────────
    ax.set_yticks(range(num_entities))
    ax.set_yticklabels(unique_entities, fontsize=TIMELINE_FONT_SIZE_YTICKS)

    # ── X-axis label ─────────────────────────────────────────────────
    ax.set_xlabel(x_label, fontsize=TIMELINE_FONT_SIZE_LABELS)
    ax.tick_params(axis="x", labelsize=TIMELINE_FONT_SIZE_TICKS)

    # ── Title ────────────────────────────────────────────────────────
    title_parts = ["Entity Lifecycle Timeline"]
    if player_filter:
        title_parts.append(f"[{player_filter}]")
    if type_filter:
        title_parts.append(f"[type: {type_filter}]")
    ax.set_title(" ".join(title_parts), fontsize=TIMELINE_FONT_SIZE_TITLE)

    # ── Legend: minimal style (no frame, small font, out of data area) ─
    # Only 5 items, well within Few's recommendation for legend entries.
    legend_items = [
        mpatches.Patch(color=TIMELINE_COLOR_STARTED, label="Started"),
        mpatches.Patch(color=TIMELINE_COLOR_COMPLETED, label="Completed"),
        mpatches.Patch(color=TIMELINE_COLOR_DESTROYED, label="Destroyed"),
        mpatches.Patch(color=TIMELINE_COLOR_CANCELLED, label="Cancelled"),
        mpatches.Patch(color=TIMELINE_COLOR_LIFESPAN, label="Lifespan"),
    ]
    ax.legend(
        handles=legend_items,
        loc="upper right",
        fontsize=TIMELINE_FONT_SIZE_LEGEND,
        frameon=False,           # Tufte: no legend border
        labelcolor="#555555",    # muted text so legend doesn't dominate
    )

    # ── Grid: x-axis only, very light dashed lines ───────────────────
    ax.grid(
        axis="x",
        color=TIMELINE_GRID_COLOR,
        alpha=TIMELINE_GRID_ALPHA,
        linewidth=0.5,
        linestyle="--",
    )

    ax.invert_yaxis()  # First entity at top
    plt.show()
    return fig

In [ ]:
# Timeline for Player 1 entities
fig_p1 = plot_entity_timeline(lifecycle_events, player_filter="p1", max_entities=50)

In [ ]:
# Timeline for Player 2 entities
fig_p2 = plot_entity_timeline(lifecycle_events, player_filter="p2", max_entities=50)

In [ ]:
# Timeline for a specific unit type (e.g., marines)
fig_marines = plot_entity_timeline(lifecycle_events, type_filter="marine")

---
## Section 5: Correlation Heatmap

Converts mixed-type entity columns to numeric, selects columns with sufficient
non-null data, and plots a correlation heatmap. Highlights unexpectedly high
correlations while warning about expected ones (e.g., health vs health_max).

In [ ]:
def build_numeric_subset(
    df,
    min_non_null_frac=0.5,
    include_economy=True,
    max_entity_cols=30,
):
    """
    Convert entity columns to numeric (coercing lifecycle strings to NaN)
    and select columns with sufficient non-null data for correlation analysis.

    Parameters:
        df (pd.DataFrame): The game state DataFrame.
        min_non_null_frac (float): Minimum fraction of non-null values for
                                   a column to be included (0.0 to 1.0).
        include_economy (bool): Whether to include economy columns
                                (p{n}_minerals, etc.).
        max_entity_cols (int): Maximum number of entity columns to include
                               (top by non-null count).

    Returns:
        pd.DataFrame: Numeric-only DataFrame suitable for correlation analysis.

    Depends on / calls:
        - ENTITY_COL_RE (module-level compiled regex)
    """
    numeric_cols = {}

    # Economy columns are already numeric
    economy_suffixes = [
        "minerals", "vespene", "supply_used", "supply_cap",
        "collection_rate_minerals", "collection_rate_vespene",
    ]
    if include_economy:
        for player in ["p1", "p2"]:
            for suffix in economy_suffixes:
                col_name = f"{player}_{suffix}"
                if col_name in df.columns:
                    numeric_cols[col_name] = pd.to_numeric(
                        df[col_name], errors="coerce"
                    )

    # Entity columns: coerce to numeric (lifecycle strings become NaN)
    entity_cols = [c for c in df.columns if ENTITY_COL_RE.match(c)]

    # Convert all entity columns to numeric and track non-null counts
    entity_numeric = {}
    entity_non_null_counts = {}
    for col in entity_cols:
        numeric_series = pd.to_numeric(df[col], errors="coerce")
        non_null_frac = numeric_series.notna().mean()
        if non_null_frac >= min_non_null_frac:
            entity_numeric[col] = numeric_series
            entity_non_null_counts[col] = non_null_frac

    # Limit to top N entity columns by non-null fraction
    top_entity_cols = sorted(
        entity_non_null_counts, key=entity_non_null_counts.get, reverse=True
    )[:max_entity_cols]

    for col in top_entity_cols:
        numeric_cols[col] = entity_numeric[col]

    result_df = pd.DataFrame(numeric_cols)
    return result_df


# Build numeric subset for correlation analysis
numeric_df = build_numeric_subset(df, min_non_null_frac=0.5, max_entity_cols=30)
print(f"Numeric subset: {numeric_df.shape[0]} rows x {numeric_df.shape[1]} columns")
print(f"Columns included: {list(numeric_df.columns)}")

In [ ]:
# ── Configurable Parameters: Correlation Heatmap ──────────────────────
# All variables use the HEATMAP_ prefix to avoid naming conflicts with
# other cells' configurable variables.

HEATMAP_FIGURE_SIZE = (14, 12)          # (width, height) in inches
HEATMAP_FONT_SIZE_TITLE = 16            # Title font size
HEATMAP_FONT_SIZE_LABELS = 12           # Axis label font size (currently unused; axes are tick-only)
HEATMAP_FONT_SIZE_TICKS = 7             # Tick label font size for both axes
HEATMAP_CMAP = "RdBu_r"                # Diverging colormap: blue-white-red centered at 0
HEATMAP_COLOR_BACKGROUND = "white"      # Figure and axes background color
HEATMAP_SPINE_COLOR = "#888888"         # Spine color (spines are removed, kept for reference)
HEATMAP_LABEL_MAX_LENGTH = 30           # Truncate column labels longer than this
HEATMAP_DPI = 100                       # Figure resolution in dots per inch
HEATMAP_COLORBAR_SHRINK = 0.8           # Colorbar height relative to axes
HEATMAP_COLORBAR_ASPECT = 30            # Colorbar width (higher = thinner)
HEATMAP_MARGIN = {                      # Replaces plt.tight_layout()
    "left": 0.18,
    "right": 0.95,
    "top": 0.94,
    "bottom": 0.18,
}


def plot_correlation_heatmap(numeric_df, threshold=0.9, figsize=(14, 12)):
    """
    Plot a correlation heatmap for the numeric DataFrame and extract
    pairs with correlation above the specified threshold.

    Styled following Tufte/Few data-visualization principles:
    - White background, no spines (heatmap cells define boundaries)
    - Diverging blue-white-red colormap centered at 0
    - Minimal colorbar chrome
    - Font sizes controlled by HEATMAP_ configurable variables
    - fig.subplots_adjust() instead of tight_layout()

    Also warns about expected high correlations (e.g., health vs health_max,
    supply_used vs supply_cap).

    Parameters:
        numeric_df (pd.DataFrame): Numeric-only DataFrame from build_numeric_subset().
        threshold (float): Absolute correlation threshold for flagging pairs.
                           Defaults to 0.9.
        figsize (tuple): Figure size for the heatmap. Overridden by
                         HEATMAP_FIGURE_SIZE configurable variable.

    Returns:
        pd.DataFrame: DataFrame of high-correlation pairs (col1, col2, correlation).

    Depends on / calls:
        - build_numeric_subset() output (numeric_df parameter)
        - HEATMAP_* configurable variables (module-level)
    """
    if numeric_df.shape[1] < 2:
        print("Not enough numeric columns for correlation analysis.")
        return pd.DataFrame()

    # Compute correlation matrix
    corr_matrix = numeric_df.corr()

    # ── Create figure with white background ──────────────────────────
    fig, ax = plt.subplots(
        figsize=HEATMAP_FIGURE_SIZE,
        dpi=HEATMAP_DPI,
        facecolor=HEATMAP_COLOR_BACKGROUND,
    )
    ax.set_facecolor(HEATMAP_COLOR_BACKGROUND)

    # ── Remove all four spines (heatmap cells define boundaries) ─────
    for spine in ax.spines.values():
        spine.set_visible(False)

    # ── Build short labels for readability ───────────────────────────
    short_labels = []
    for col in corr_matrix.columns:
        # Truncate long column names to keep tick labels compact
        if len(col) <= HEATMAP_LABEL_MAX_LENGTH:
            label = col
        else:
            label = col[: HEATMAP_LABEL_MAX_LENGTH - 3] + "..."
        short_labels.append(label)

    # ── Draw the heatmap ─────────────────────────────────────────────
    # Diverging blue-white-red palette centered at 0 for correlation data
    sns.heatmap(
        corr_matrix,
        xticklabels=short_labels,
        yticklabels=short_labels,
        cmap=HEATMAP_CMAP,
        center=0,
        vmin=-1,
        vmax=1,
        annot=False,          # Too many columns for per-cell annotation
        linewidths=0.3,       # Thin lines between cells for subtle grid
        linecolor="#dddddd",  # Light gray cell dividers
        ax=ax,
        # Colorbar styling: thin, minimal chrome
        cbar_kws={
            "shrink": HEATMAP_COLORBAR_SHRINK,
            "aspect": HEATMAP_COLORBAR_ASPECT,
        },
    )

    # ── Style the colorbar minimally ─────────────────────────────────
    # The colorbar is the second axes object created by sns.heatmap
    colorbar = ax.collections[0].colorbar
    colorbar.outline.set_visible(False)                  # Remove colorbar border
    colorbar.ax.tick_params(labelsize=HEATMAP_FONT_SIZE_TICKS, length=3, width=0.5)

    # ── Title and tick labels ────────────────────────────────────────
    ax.set_title(
        f"Correlation Heatmap (threshold={threshold})",
        fontsize=HEATMAP_FONT_SIZE_TITLE,
        pad=12,
        fontweight="normal",
    )

    # X-axis: 45° rotation since column names are long entity identifiers
    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation=45,
        ha="right",
        fontsize=HEATMAP_FONT_SIZE_TICKS,
    )

    # Y-axis: horizontal labels for readability (Tufte: all text readable)
    ax.set_yticklabels(
        ax.get_yticklabels(),
        rotation=0,
        fontsize=HEATMAP_FONT_SIZE_TICKS,
    )

    # ── Layout via explicit margins (replaces tight_layout) ──────────
    fig.subplots_adjust(**HEATMAP_MARGIN)
    plt.show()

    # ── Extract high-correlation pairs ───────────────────────────────
    high_corr_pairs = []
    cols = corr_matrix.columns
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            corr_val = corr_matrix.iloc[i, j]
            if abs(corr_val) >= threshold:
                high_corr_pairs.append({
                    "col1": cols[i],
                    "col2": cols[j],
                    "correlation": round(corr_val, 4),
                })

    high_corr_df = pd.DataFrame(high_corr_pairs)
    if not high_corr_df.empty:
        high_corr_df = high_corr_df.sort_values(
            "correlation", ascending=False, key=abs
        ).reset_index(drop=True)

    # Warn about expected high correlations
    expected_patterns = [
        ("health", "health_max", "health tracks health_max"),
        ("supply_used", "supply_cap", "supply columns are correlated by game design"),
        ("minerals", "collection_rate_minerals", "income drives stockpile"),
    ]

    if not high_corr_df.empty:
        print(f"\nHigh-correlation pairs (|r| >= {threshold}): {len(high_corr_df)}")
        print()
        for _, row in high_corr_df.iterrows():
            # Check if this is an expected correlation
            is_expected = False
            for pat1, pat2, reason in expected_patterns:
                if (pat1 in row["col1"] and pat2 in row["col2"]) or \
                   (pat2 in row["col1"] and pat1 in row["col2"]):
                    is_expected = True
                    print(
                        f"  [EXPECTED] {row['col1']} <-> {row['col2']}: "
                        f"{row['correlation']:.4f} ({reason})"
                    )
                    break
            if not is_expected:
                print(
                    f"  {row['col1']} <-> {row['col2']}: {row['correlation']:.4f}"
                )
    else:
        print(f"No pairs with |correlation| >= {threshold}")

    return high_corr_df


# Plot the heatmap and show high-correlation pairs
high_corr_pairs = plot_correlation_heatmap(numeric_df, threshold=0.9)

---
## Section 6: Data Sparsity Visualization

Shows where real data exists vs NaN or lifecycle strings across all entity
columns over game time. Groups by entity to handle the 1600+ column count,
and samples game_loops for performance.

In [ ]:
# ── Configurable Parameters: Data Sparsity ──────────────────────────
# All visual constants for plot_data_sparsity() are defined here so they
# can be tweaked in one place without modifying the function body.
# The SPARSITY_ prefix avoids naming conflicts with other cells.

SPARSITY_FIGURE_SIZE = (16, 10)         # (width, height) in inches
SPARSITY_FONT_SIZE_TITLE = 16           # Title font size
SPARSITY_FONT_SIZE_LABELS = 12          # Axis label font size (x and y axis labels)
SPARSITY_FONT_SIZE_TICKS = 10           # X-axis tick label font size
SPARSITY_FONT_SIZE_YTICKS = 6           # Y-axis tick label font size (entity names)
SPARSITY_CMAP = "YlGn"                 # Sequential colormap (pale=low, dark=high)
SPARSITY_COLOR_BACKGROUND = "white"     # Figure and axes background color
SPARSITY_SPINE_COLOR = "#888888"        # Left/bottom spine color (thin gray)
SPARSITY_CBAR_SHRINK = 0.8             # Colorbar height relative to axes
SPARSITY_DPI = 100                      # Figure resolution in dots per inch
SPARSITY_MARGIN = {                     # Replaces plt.tight_layout()
    "left": 0.10,
    "right": 0.95,
    "top": 0.94,
    "bottom": 0.12,
}


def plot_data_sparsity(
    df,
    entity_catalog,
    player_filter="p1",
    sample_every_n=10,
    figsize=(16, 10),
):
    """
    Visualize data sparsity across game time for one player's entities.

    Renders a heatmap styled with Tufte/Few data-visualization principles:
    white background, no top/right spines, thin gray remaining spines,
    sequential single-hue colormap (pale = low data, dark = high data),
    and a minimal colorbar. Font sizes and colors are controlled by
    SPARSITY_ configurable variables defined at the top of this cell.

    X-axis: game_loop (sampled every Nth for performance).
    Y-axis: entities (one row per entity, grouped by type).
    Color: fraction of the entity's attribute columns that contain real
           numeric data (vs NaN or lifecycle strings) at each game_loop.

    Parameters:
        df (pd.DataFrame): The game state DataFrame.
        entity_catalog (dict): Output from build_entity_catalog().
        player_filter (str): Which player to show ("p1" or "p2").
        sample_every_n (int): Sample every Nth game_loop for performance.
                              Higher values = faster but coarser.
        figsize (tuple): Figure size. Overridden by SPARSITY_FIGURE_SIZE
                         configurable variable.

    Returns:
        matplotlib.figure.Figure: The created figure.

    Depends on / calls:
        - build_entity_catalog() output (entity_catalog parameter)
        - ALL_LIFECYCLE_STRINGS (module-level frozenset)
        - SPARSITY_* configurable variables (module-level)
    """
    # Filter entities for the requested player
    player_entities = {
        prefix: info
        for prefix, info in entity_catalog.items()
        if info["player"] == player_filter
    }

    if not player_entities:
        print(f"No entities found for {player_filter}")
        return None

    # Sort entities by type then seq_id for visual grouping
    sorted_prefixes = sorted(
        player_entities.keys(),
        key=lambda p: (player_entities[p]["entity_type"], player_entities[p]["seq_id"]),
    )

    # Sample game_loops for performance
    sampled_df = df.iloc[::sample_every_n].copy()
    game_loops = sampled_df["game_loop"].values
    n_loops = len(game_loops)
    n_entities = len(sorted_prefixes)

    # Build the sparsity matrix: rows = entities, columns = sampled game_loops
    # Value = fraction of attribute columns with real (non-NaN, non-lifecycle) data
    sparsity_matrix = np.full((n_entities, n_loops), np.nan)

    for ent_idx, prefix in enumerate(sorted_prefixes):
        info = player_entities[prefix]
        entity_cols = info["columns"]
        n_attrs = len(entity_cols)

        if n_attrs == 0:
            continue

        # For each sampled row, count how many attribute columns have real data
        # "Real data" means: not NaN AND not a lifecycle string
        for loop_idx, row_iloc in enumerate(range(0, len(df), sample_every_n)):
            if row_iloc >= len(df):
                break

            real_count = 0
            for col in entity_cols:
                val = df[col].iloc[row_iloc]
                # Check: not NaN/None AND not a lifecycle string
                if pd.notna(val):
                    if not (isinstance(val, str) and val in ALL_LIFECYCLE_STRINGS):
                        real_count += 1

            sparsity_matrix[ent_idx, loop_idx] = real_count / n_attrs

    # ── Figure Setup (Tufte: white background, no chartjunk) ─────────
    fig, ax = plt.subplots(
        figsize=SPARSITY_FIGURE_SIZE,
        dpi=SPARSITY_DPI,
        facecolor=SPARSITY_COLOR_BACKGROUND,
    )
    ax.set_facecolor(SPARSITY_COLOR_BACKGROUND)

    # ── Remove Chartjunk: hide top/right spines, style remaining ones ─
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(SPARSITY_SPINE_COLOR)
    ax.spines["bottom"].set_color(SPARSITY_SPINE_COLOR)
    ax.spines["left"].set_linewidth(0.5)
    ax.spines["bottom"].set_linewidth(0.5)

    # ── Tick Styling ────────────────────────────────────────────────
    ax.tick_params(
        axis="both", which="both", labelsize=SPARSITY_FONT_SIZE_TICKS,
        colors=SPARSITY_SPINE_COLOR, length=3, width=0.5,
    )

    # ── Heatmap via imshow (sequential colormap: pale=low, dark=high) ─
    im = ax.imshow(
        sparsity_matrix,
        aspect="auto",
        cmap=SPARSITY_CMAP,
        vmin=0,
        vmax=1,
        interpolation="nearest",
    )

    # ── X-axis: game_loop values at intervals, 45° rotation ─────────
    x_tick_step = max(1, n_loops // 10)
    ax.set_xticks(range(0, n_loops, x_tick_step))
    ax.set_xticklabels(
        [str(game_loops[i]) for i in range(0, n_loops, x_tick_step)],
        rotation=45,
        ha="right",
        fontsize=SPARSITY_FONT_SIZE_TICKS,
    )
    ax.set_xlabel("Game Loop", fontsize=SPARSITY_FONT_SIZE_LABELS)

    # ── Y-axis: entity names (short form for readability) ────────────
    # Use entity_type + seq_id as short label, horizontal for readability
    short_labels = [
        f"{player_entities[p]['entity_type']}_{player_entities[p]['seq_id']}"
        for p in sorted_prefixes
    ]
    ax.set_yticks(range(n_entities))
    ax.set_yticklabels(
        short_labels,
        fontsize=SPARSITY_FONT_SIZE_YTICKS,
        rotation=0,
    )
    ax.set_ylabel("Entity", fontsize=SPARSITY_FONT_SIZE_LABELS)

    # ── Title ────────────────────────────────────────────────────────
    ax.set_title(
        f"Data Sparsity: {player_filter} entities "
        f"(sampled every {sample_every_n} game_loops)",
        fontsize=SPARSITY_FONT_SIZE_TITLE,
        pad=12,
        fontweight="normal",
    )

    # ── Colorbar: minimal chrome ─────────────────────────────────────
    cbar = plt.colorbar(im, ax=ax, shrink=SPARSITY_CBAR_SHRINK)
    cbar.set_label(
        "Fraction of attributes with real data",
        fontsize=SPARSITY_FONT_SIZE_LABELS,
    )
    cbar.outline.set_visible(False)                     # Remove colorbar border
    cbar.ax.tick_params(
        labelsize=SPARSITY_FONT_SIZE_TICKS,
        length=3,
        width=0.5,
        colors=SPARSITY_SPINE_COLOR,
    )

    # ── Layout via explicit margins (replaces tight_layout) ──────────
    fig.subplots_adjust(**SPARSITY_MARGIN)
    plt.show()
    return fig

In [ ]:
# Data sparsity for Player 1
fig_sparsity_p1 = plot_data_sparsity(
    df, entity_catalog, player_filter="p1", sample_every_n=20
)

In [ ]:
# Data sparsity for Player 2
fig_sparsity_p2 = plot_data_sparsity(
    df, entity_catalog, player_filter="p2", sample_every_n=20
)